In [ ]:
%%capture
!pip install unsloth
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
import unsloth
print("Unsloth OK:", unsloth.__version__)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth OK: 2026.4.4


In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = 4096,   # UPDATED: was 2048 — needed for full-document training entries
    dtype = None,
    load_in_4bit = True,
)
print("Model loaded OK")

==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


Model loaded OK


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)
print("LoRA adapters attached OK")

Unsloth 2026.4.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


LoRA adapters attached OK


In [ ]:
from google.colab import files
import json
from datasets import Dataset

uploaded = files.upload()  # Select dataset_final.json when prompted

with open("dataset_final.json", "r") as f:
    raw_data = json.load(f)

print(f"Loaded {len(raw_data)} entries")

def format_entry(entry):
    return {
        "text": (
            "<|begin_of_text|>"
            "<|start_header_id|>system<|end_header_id|>
"
            "You are Lura-Rent, an expert AI Legal Auditor specializing in Indian "
            "Residential Rental Agreements under the Maharashtra Rent Control Act, 1999. "
            "Analyze the given clause or document and provide a structured risk assessment "
            "with plain language summary, legal assessment, tenant advisory, and next steps "
            "where applicable.<|eot_id|>"
            "<|start_header_id|>user<|end_header_id|>
"
            f"Analyze this clause:
{entry['clause']}<|eot_id|>"
            "<|start_header_id|>assistant<|end_header_id|>
"
            f"{entry['analysis']}<|eot_id|>"
        )
    }

formatted = [format_entry(e) for e in raw_data]
dataset = Dataset.from_list(formatted)
print(f"Dataset formatted OK — {len(dataset)} entries ready")

Saving dataset.json to dataset.json
Loaded 114 entries
Dataset formatted OK — 114 entries ready


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 4096,   # UPDATED: was 2048
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 10,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "outputs",
    ),
)

print("Starting training...")
trainer_stats = trainer.train()
print(f"Done! Final loss: {trainer_stats.training_loss:.4f}")

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/114 [00:00<?, ? examples/s]

Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 114 | Num Epochs = 10 | Total steps = 150
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,2.666753
20,1.572789
30,1.356460
40,1.086423
50,0.932516
60,0.761724
70,0.630116
80,0.473197
90,0.444657
100,0.313330


Done! Final loss: 0.7473


In [ ]:
model.save_pretrained("lura-rent-lora")
tokenizer.save_pretrained("lura-rent-lora")
print("LoRA saved OK")

LoRA saved OK


In [ ]:
model.save_pretrained_gguf(
    "lura-rent-gguf",
    tokenizer,
    quantization_method = "q4_k_m"
)
print("GGUF export complete!")

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [02:47<00:00, 83.64s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [01:44<00:00, 52.43s/it]


Unsloth: Merge process complete. Saved to `/content/lura-rent-gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['lura-rent-gguf_gguf/llama-3.2-3b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['lura-rent-gguf_gguf/llama-3.2-3b-instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model lura-rent-gguf_gguf/llama-3.2-3b-instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to lura-rent-gguf_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f lura-rent-gguf_gguf/Modelfile
GGUF export complete!


In [ ]:
from google.colab import files
import shutil

shutil.make_archive("lura-rent-lora", "zip", "lura-rent-lora")
files.download("lura-rent-lora.zip")
files.download("lura-rent-gguf_gguf/llama-3.2-3b-instruct.Q4_K_M.gguf")
files.download("lura-rent-gguf_gguf/Modelfile")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
trainer2 = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 4096,   # UPDATED: was 2048
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 10,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "outputs",
    ),
)

print("Running extended training...")
trainer_stats2 = trainer2.train()
print("Done!")
print(f"Final training loss: {trainer_stats2.training_loss:.4f}")

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/77 [00:00<?, ? examples/s]

Running extended training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 77 | Num Epochs = 10 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Step,Training Loss
5,1.334483
10,1.157191
15,0.941810
20,0.916899
25,0.628217
30,0.744735
35,0.422765
40,0.511252
45,0.325036
50,0.301129


Done!
Final training loss: 0.4218


In [ ]:
model.save_pretrained("lura-rent-lora")
tokenizer.save_pretrained("lura-rent-lora")
print("LoRA adapters saved OK")

LoRA adapters saved OK


In [ ]:
model.save_pretrained_gguf(
    "lura-rent-gguf",
    tokenizer,
    quantization_method = "q4_k_m"
)
print("GGUF export complete!")

Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.



Unsloth: Preparing safetensor model files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [02:56<00:00, 88.35s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [02:06<00:00, 63.01s/it]


Unsloth: Merge process complete. Saved to `/content/lura-rent-gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes


In [ ]:
from google.colab import files
import shutil

shutil.make_archive("lura-rent-lora", "zip", "lura-rent-lora")
files.download("lura-rent-lora.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
%%javascript
function KeepAlive() {
    document.querySelector("#connect")?.click();
    console.log("Keeping Colab alive...");
}
setInterval(KeepAlive, 60000);

<IPython.core.display.Javascript object>

In [ ]:
from unsloth import FastLanguageModel
from safetensors.torch import load_file
from peft import set_peft_model_state_dict
import zipfile, torch

# Step 1: Reload base model fresh
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = 4096,   # UPDATED: was 2048
    dtype = None,
    load_in_4bit = True,
)

# Step 2: Attach LoRA structure
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

# Step 3: Load your trained weights into the LoRA structure
state_dict = load_file("lura-rent-lora/adapter_model.safetensors")
set_peft_model_state_dict(model, state_dict)

print("Model + LoRA ready. Exporting to GGUF...")

# Step 4: Export immediately
model.save_pretrained_gguf(
    "lura-rent-gguf",
    tokenizer,
    quantization_method = "q4_k_m"
)
print("GGUF export complete!")

==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth 2026.4.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Model + LoRA ready. Exporting to GGUF...
Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

...


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [01:01<00:00, 30.94s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [01:29<00:00, 44.96s/it]


Unsloth: Merge process complete. Saved to `/content/lura-rent-gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['lura-rent-gguf_gguf/llama-3.2-3b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['lura-rent-gguf_gguf/llama-3.2-3b-instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model lura-rent-gguf_gguf/llama-3.2-3b-instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to lura-rent-gguf_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f lura-rent-gguf_gguf/Modelfile
GGUF export complete!


In [ ]:
from google.colab import files
files.download("lura-rent-gguf_gguf/llama-3.2-3b-instruct.Q4_K_M.gguf")
files.download("lura-rent-gguf_gguf/Modelfile")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>